# Persistence and Checkpointers

**What you will build:** durable memory for a graph. In 0.6 and 1.4 *you* carried the message list forward by hand. LangGraph does it for you with a **checkpointer**: attach one, give each conversation a `thread_id`, and the graph remembers automatically — even across separate calls.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/23_persistence_and_checkpointers.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## A chatbot graph, then make it remember

Start with a one-node chat graph that has *no* memory — each call is independent (the statelessness of 0.1).

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState

def chat(state: MessagesState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

plain = builder.compile()   # no checkpointer -> no memory
plain.invoke({"messages": [{"role": "user", "content": "My name is Sam."}]})
r = plain.invoke({"messages": [{"role": "user", "content": "What's my name?"}]})
print("no memory:", r["messages"][-1].content)   # it doesn't know

## Add a checkpointer

Compile the *same* graph with an `InMemorySaver`. Now every call tagged with the same `thread_id` shares state — the checkpointer saves the messages after each turn and reloads them on the next.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_graph = builder.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "conversation-1"}}
memory_graph.invoke({"messages": [{"role": "user", "content": "My name is Sam."}]}, config=cfg)
r = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=cfg)
print("with memory:", r["messages"][-1].content)   # now it knows

Same graph, same code — the only difference is the checkpointer and the `thread_id`. Persistence is a property you *attach*, not something you thread through every function. That's a real step up from carrying `message_history` by hand.

## Threads are separate conversations

A different `thread_id` is a clean slate — this is how one app serves many users without their chats mixing.

In [ ]:
other = {"configurable": {"thread_id": "conversation-2"}}
r = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=other)
print("different thread:", r["messages"][-1].content)   # no idea - separate conversation

::::{dropdown} 🔍 Where the state lives, and time travel
:color: info

`InMemorySaver` keeps checkpoints in RAM (great for notebooks; gone when the kernel restarts). Swap it for `SqliteSaver` or `PostgresSaver` and the same graph persists to disk or a database — with no change to your nodes.

Because every step is checkpointed, you can inspect and even rewind history:

```python
snapshot = memory_graph.get_state(cfg)                 # current state
for s in memory_graph.get_state_history(cfg):          # every past checkpoint
    print(len(s.values['messages']), 'messages')
```

This *time-travel* is something you'd have to build entirely yourself in Block 0 — it's one of the concrete reasons to use LangGraph for stateful agents.
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **The agent doesn't remember** | No checkpointer, or a new `thread_id` each call | Compile with a checkpointer; reuse the same `thread_id` per conversation |
| **All users share one memory** | Same `thread_id` for everyone | Give each user/conversation its own `thread_id` |
| **Memory lost on kernel restart** | `InMemorySaver` is RAM-only | Use `SqliteSaver`/`PostgresSaver` for durable storage |
::::

**What's next:** in **2.4** we use the checkpointer for **human-in-the-loop** — pausing a graph mid-run to wait for a person, then resuming.